In [1]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, mean, stddev

print("🚀 Initializing PySpark with Apache Iceberg extensions...")

# Bootstrap the Spark session to read our WAP-managed Iceberg warehouse
spark = (SparkSession.builder
    .appName("ML_Demand_Forecasting_Sandbox")
    .config("spark.sql.catalog.nyc", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nyc.type", "hadoop")
    .config("spark.sql.catalog.nyc.warehouse", "/opt/airflow/data/warehouse")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .getOrCreate())

print("✅ Spark Session ready! Catalog 'nyc' is successfully mounted.")

🚀 Initializing PySpark with Apache Iceberg extensions...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 02:52:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Session ready! Catalog 'nyc' is successfully mounted.


In [20]:
print("📊 Verifying connection to the Gold ML layer...")

# Fetch a small slice of our hourly observation grid
gold_ml_df = spark.table("nyc.gold.hourly_ml_observations")

# Use our Pandas formatting trick for clean Jupyter rendering
display(gold_ml_df.orderBy(col("pickup_hour_ts").desc()).limit(5).toPandas())

📊 Verifying connection to the Gold ML layer...


,target_year,target_month,pickup_location_id,pickup_hour_ts,ride_count
0,2026,5,26,2026-05-31 23:00:00,0
1,2026,5,13,2026-05-31 23:00:00,2
2,2026,5,30,2026-05-31 23:00:00,0
3,2026,5,83,2026-05-31 23:00:00,1
4,2026,5,57,2026-05-31 23:00:00,0


In [10]:
print("🎯 Isolating the highest-volume target location...")

# Group by location and sum the total trips to find our highest-volume hub
top_location_row = (gold_ml_df
    # .filter(col("ride_count") > 5)
    .groupBy("pickup_location_id")
    .sum("ride_count")
    .orderBy(col("sum(ride_count)").desc())
    .first())

target_location_id = top_location_row["pickup_location_id"]
total_historical_trips = top_location_row["sum(ride_count)"]

print(f"🏆 Target Location Identified: ID {target_location_id}")
print(f"📈 Total Historical Trips: {total_historical_trips:,}")

# Filter our main dataframe down to ONLY this specific location
df_target_zone = gold_ml_df.filter(col("pickup_location_id") == target_location_id)

🎯 Isolating the highest-volume target location...
🏆 Target Location Identified: ID 237
📈 Total Historical Trips: 490,090


Previously, next hour ride demand -- predict actual ride count -- Upper East Side South -- target 300 298.455 
Signal -- high/surge, normal demand, low 
Just take the top location to simplify training. 
train a separate model for each location. 



 

In [22]:
from pyspark.sql import functions as F
gold_ml_df = spark.table("nyc.gold.hourly_ml_observations")

bounds = gold_ml_df.agg(
    F.min("target_year").alias("min_year"),
    F.max("target_year").alias("max_year"),
    F.min("target_month").alias("min_month"),
    F.max("target_month").alias("max_month")

)

bounds.show()

+--------+--------+---------+---------+
|min_year|max_year|min_month|max_month|
+--------+--------+---------+---------+
|    2025|    2026|        2|       11|
+--------+--------+---------+---------+



In [4]:
from pyspark.sql.functions import min, max, concat, lit, lpad

gold_ml_df = spark.table("nyc.gold.hourly_ml_observations")

df_with_date = gold_ml_df.withColumn(
    "year_month", 
    concat(gold_ml_df["target_year"], lit("-"), lpad(gold_ml_df["target_month"], 2, '0'))
)
df_with_date.select("year_month").distinct().orderBy(col("year_month")).show()

+----------+
|year_month|
+----------+
|   2025-01|
|   2025-02|
|   2025-08|
+----------+



In [33]:
gold_ml_df = spark.table("nyc.gold.hourly_ml_observations")

df_with_date = gold_ml_df.withColumn(
    "year_month", 
    concat(gold_ml_df["target_year"], lit("-"), lpad(gold_ml_df["target_month"], 2, '0'))
)
df_with_date.select("year_month").distinct().orderBy(col("year_month")).show()

[Stage 53:===================>                                      (1 + 2) / 3]

+----------+
|year_month|
+----------+
|   2025-02|
|   2025-03|
|   2025-04|
|   2025-05|
|   2025-11|
|   2026-02|
|   2026-03|
|   2026-04|
|   2026-05|
+----------+

